In [0]:
from pyspark.sql.functions import *

In [0]:
source_path = "abfss://source@mvnazureproject.dfs.core.windows.net/"

bronze_path = "abfss://bronze@mvnazureproject.dfs.core.windows.net/health_claims"

checkpoint_path = "abfss://bronze@mvnazureproject.dfs.core.windows.net/checkpoint/health_claims"

schema_path = "abfss://bronze@mvnazureproject.dfs.core.windows.net/schema/health_claims"

In [0]:
bronze_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", schema_path)
         .load(source_path)
)

In [0]:
# Write to Bronze
(
    bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .start(bronze_path)
        .awaitTermination()
)

In [0]:
%sql
SELECT *
FROM mvn_db_catalog.bronze.health_claims
ORDER BY ClaimID;

In [0]:
%sql
SELECT COUNT(*) FROM mvn_db_catalog.bronze.health_claims;

In [0]:
%sql
SELECT COUNT(*) FROM mvn_db_catalog.silver.health_claims;

In [0]:
%sql
SELECT COUNT(*) FROM mvn_db_catalog.gold.fact_claims;

In [0]:
%sql
SELECT * FROM mvn_db_catalog.gold.fact_claims
ORDER BY ClaimID;

In [0]:
%sql
SELECT COUNT(*)
FROM mvn_db_catalog.bronze.health_claims;

In [0]:
%sql
SELECT *
FROM mvn_db_catalog.silver.health_claims
ORDER BY ClaimID;